In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import sys, importlib
sys.path.append("/content/drive/MyDrive/ft_vs_rag_project")

from ft_vs_rag_improved_pipeline import (
    DATA_DIR, INDEX_DIR_FULL, INDEX_DIR_DYNAMIC,
    hotpot_extract, save_jsonl, make_dynamic_split,
    build_faiss_index_from_jsonl
)

print("OK ✅", DATA_DIR)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OK ✅ /content/ft_vs_rag_multidata/data_hotpot


In [ ]:
# ===========================
# CPU: data prep only
# ===========================
# Optional: keep project data on Drive (adjust path as you like)
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/ft_vs_rag_project"
!mkdir -p "$DRIVE_PROJECT_DIR"

# ---------------------------
# Imports (no embeddings here)
# ---------------------------
from ft_vs_rag_improved_pipeline import (
    DATA_DIR,  # default: /content/ft_vs_rag_multidata/data_hotpot
    hotpot_extract, save_jsonl, make_dynamic_split,
    create_pretraining_dataset
)

# CPU-friendly chunking to reduce number of chunks
CHUNK_SIZE = 768
OVERLAP    = 16

# 1) Extract HotpotQA splits (no embeddings)
docs_train, qas_train = hotpot_extract(
    split="train", variant="distractor",
    sample_size=None, chunk_size=CHUNK_SIZE, overlap=OVERLAP
)
docs_dev, qas_dev = hotpot_extract(
    split="validation", variant="distractor",
    sample_size=None, chunk_size=CHUNK_SIZE, overlap=OVERLAP
)

# 2) Stable vs Dynamic (by title membership in train)
train_titles = {d["title"] for d in docs_train}
stable_docs  = list(docs_train)
dynamic_docs = [d for d in docs_dev if d["title"] not in train_titles]

# 3) QA partitions (for reporting; training uses only train_ft_qas)
train_ft_qas     = [q for q in qas_train if all(t in  train_titles for t in q["supporting_docs"])]
test_dynamic_qas = [q for q in qas_dev   if all(t not in train_titles for t in q["supporting_docs"])]
test_stable_qas  = [q for q in qas_dev   if all(t in  train_titles for t in q["supporting_docs"])]

# 4) Save initial JSONL files (CPU)
save_jsonl(DATA_DIR / "stable_docs.jsonl", stable_docs)
save_jsonl(DATA_DIR / "dynamic_docs.jsonl", dynamic_docs)
save_jsonl(DATA_DIR / "train_ft_qas.jsonl", train_ft_qas)
save_jsonl(DATA_DIR / "test_dynamic_qas.jsonl", test_dynamic_qas)  # optional reporting
save_jsonl(DATA_DIR / "test_stable_qas.jsonl",  test_stable_qas)

print(f"Saved {len(stable_docs)} stable docs and {len(dynamic_docs)} dynamic docs.")
print(f"Training QA: {len(train_ft_qas)} | Dev dynamic QA: {len(test_dynamic_qas)} | Dev stable QA: {len(test_stable_qas)}")

# 5) Dynamic split → indexed vs held-out (unseen) [no embeddings]
split_paths = make_dynamic_split(
    data_dir=DATA_DIR,
    dynamic_docs_name="dynamic_docs.jsonl",
    dev_qas_name="test_dynamic_qas.jsonl",
    ratio_indexed=0.8,
    seed=42
)
print("Dynamic split files:", split_paths)

# 6) Create pretraining dataset (no embeddings)
create_pretraining_dataset(
    data_dir=DATA_DIR,
    input_filename="stable_docs.jsonl",
    output_filename="docs_for_pretrain.jsonl"
)
print("Wrote docs_for_pretrain.jsonl")

# 7) (Optional) copy prepared data to Drive for the GPU notebook
!rsync -ah --info=NAME1,STATS1 "/content/ft_vs_rag_multidata/" "$DRIVE_PROJECT_DIR/ft_vs_rag_multidata/"
print("Data synced to Drive:", DRIVE_PROJECT_DIR)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

distractor/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/validation-00000-of-00001.par(…):   0%|          | 0.00/27.5M [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Saved 533355 stable docs and 25481 dynamic docs.
Training QA: 90447 | Dev dynamic QA: 775 | Dev stable QA: 4248
Dynamic split files: {'dynamic_indexed_docs': PosixPath('/content/ft_vs_rag_multidata/data_hotpot/dynamic_indexed_docs.jsonl'), 'dynamic_test_docs': PosixPath('/content/ft_vs_rag_multidata/data_hotpot/dynamic_test_docs.jsonl'), 'test_qas_unseen': PosixPath('/content/ft_vs_rag_multidata/data_hotpot/test_qas_unseen.jsonl'), 'test_qas_seen_dynamic': PosixPath('/content/ft_vs_rag_multidata/data_hotpot/test_qas_seen_dynamic.jsonl')}
Wrote docs_for_pretrain.jsonl
created directory /content/drive/MyDrive/ft_vs_rag_project/ft_vs_rag_multidata
./
data_hotpot/
data_hotpot/docs_for_pretrain.jsonl
data_hotpot/dynamic_docs.jsonl
data_hotpot/dynamic_indexed_docs.jsonl
data_hotpot/dynamic_test_docs.jsonl
data_hotpot/stable_docs.jsonl
data_hotpot/test_dynamic_qas.jsonl
data_hotpot/test_qas_seen_dynamic.jsonl
data_hotpot/test_qas_unseen.jsonl
data_hotpot/test_stable_qas.jsonl
data_hotpot/trai

# TEMP

# Copy common functions to google drive (ft_vs_rag_improved_pipeline)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers sentence-transformers faiss-cpu datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 52.1 MB/s eta 0:00:00


In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/ft_vs_rag_project")

In [ ]:
#!rm /content/ft_vs_rag_improved_pipeline.py

In [ ]:
from pathlib import Path

clean_code = r"""
from __future__ import annotations
from pathlib import Path
from typing import Dict, List, Union, Tuple, Set
import json
import random

# ============================================================
# CONSTANTS (paths organization)
# ============================================================

PROJECT_ROOT = Path("/content")
DATA_DIR = PROJECT_ROOT / "ft_vs_rag_multidata" / "data_hotpot"
INDEX_DIR_FULL = PROJECT_ROOT / "ft_vs_rag_multidata" / "faiss_index_full"
INDEX_DIR_DYNAMIC = PROJECT_ROOT / "ft_vs_rag_multidata" / "faiss_index_dynamic"
AXOLOTL_CONFIG_DIR = PROJECT_ROOT / "axolotl_configs"

for p in [DATA_DIR, INDEX_DIR_FULL, INDEX_DIR_DYNAMIC, AXOLOTL_CONFIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ============================================================
# JSONL utilities
# ============================================================

def read_jsonl(path: Union[str, Path]) -> List[dict]:
    rows: List[dict] = []
    with Path(path).open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def save_jsonl(path: Union[str, Path], rows: List[dict]) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return path


# ============================================================
# HOTPOT QA processing + chunking
# ============================================================

def chunk_text(text: str, tokenizer, chunk_size: int = 512, overlap: int = 32) -> List[str]:
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks: List[str] = []
    i = 0
    while i < len(tokens):
        chunk_tokens = tokens[i:i+chunk_size]
        chunk = tokenizer.decode(chunk_tokens).strip()
        chunks.append(chunk)
        i += max(1, chunk_size - overlap)
    return chunks


def hotpot_extract(
    split: str = "train",
    variant: str = "distractor",
    sample_size: int | None = None,
    tokenizer_name: str = "mistralai/Mistral-7B-Instruct-v0.2",
    chunk_size: int = 512,
    overlap: int = 32,
) -> Tuple[List[dict], List[dict]]:
    '''
    Extract HotpotQA, build document chunks, return (docs, qas)

    docs item: {"doc_id": "<title>::chunk<i>", "title": <title>, "text": <chunk>}
    qas  item: {"id": ..., "question": ..., "answer": ..., "supporting_docs": [titles...]}
    '''
    from datasets import load_dataset
    from transformers import AutoTokenizer

    ds = load_dataset("hotpot_qa", variant, split=split)
    if sample_size is not None:
        ds = ds.select(range(sample_size))

    tok = AutoTokenizer.from_pretrained(tokenizer_name)

    docs: Dict[str, str] = {}
    qas: List[dict] = []
    for ex in ds:
        for title, sents in zip(ex["context"]["title"], ex["context"]["sentences"]):
            t = (title or "").strip()
            docs.setdefault(t, "")
            docs[t] += " " + " ".join(sents)

        qas.append({
            "id": ex.get("_id") or ex.get("id"),
            "question": ex.get("question", ""),
            "answer": ex.get("answer", ""),
            "supporting_docs": list(set((t or "").strip() for t in ex["supporting_facts"]["title"]))
        })

    doc_list: List[dict] = []
    for title, full_text in docs.items():
        full_text = full_text.strip()
        for i, chunk in enumerate(chunk_text(full_text, tok, chunk_size, overlap)):
            doc_list.append({"doc_id": f"{title}::chunk{i}", "title": title, "text": chunk})

    return doc_list, qas


# ============================================================
# DYNAMIC SPLIT (keep an unseen test set not indexed nor fine-tuned)
# ============================================================

def split_dynamic_titles(dynamic_docs: List[dict], ratio_indexed: float = 0.8, seed: int = 42) -> Tuple[Set[str], Set[str]]:
    random.seed(seed)
    titles = sorted({(d.get("title") or "").strip() for d in dynamic_docs if d.get("title")})
    random.shuffle(titles)
    cut = int(len(titles) * ratio_indexed)
    return set(titles[:cut]), set(titles[cut:])

def filter_qas_by_titles(qas: List[dict], allowed_titles: Set[str]) -> List[dict]:
    out: List[dict] = []
    for q in qas:
        supp = {(t or "").strip() for t in q.get("supporting_docs", []) if t}
        if supp and supp.issubset(allowed_titles):
            out.append(q)
    return out

def make_dynamic_split(
    data_dir: Union[str, Path] = DATA_DIR,
    dynamic_docs_name: str = "dynamic_docs.jsonl",
    dev_qas_name: str = "test_dynamic_qas.jsonl",
    ratio_indexed: float = 0.8,
    seed: int = 42
) -> Dict[str, Path]:
    '''
    Produce:
      - dynamic_indexed_docs.jsonl  (goes to RAG index)
      - dynamic_test_docs.jsonl     (held-out; invisible to RAG & FT)
      - test_qas_unseen.jsonl       (QAs whose supporting docs are all held-out)
      - test_qas_seen_dynamic.jsonl (QAs supported only by indexed dynamic titles)
    '''
    data_dir = Path(data_dir)
    dynamic_docs = read_jsonl(data_dir / dynamic_docs_name)
    dev_qas      = read_jsonl(data_dir / dev_qas_name)

    indexed_titles, test_titles = split_dynamic_titles(dynamic_docs, ratio_indexed=ratio_indexed, seed=seed)

    dyn_indexed_docs = [d for d in dynamic_docs if (d.get("title") or "").strip() in indexed_titles]
    dyn_test_docs    = [d for d in dynamic_docs if (d.get("title") or "").strip() in test_titles]

    test_qas_unseen        = filter_qas_by_titles(dev_qas, test_titles)
    test_qas_seen_dynamic  = filter_qas_by_titles(dev_qas, indexed_titles)

    out: Dict[str, Path] = {}
    out["dynamic_indexed_docs"]  = save_jsonl(data_dir / "dynamic_indexed_docs.jsonl", dyn_indexed_docs)
    out["dynamic_test_docs"]     = save_jsonl(data_dir / "dynamic_test_docs.jsonl",    dyn_test_docs)
    out["test_qas_unseen"]       = save_jsonl(data_dir / "test_qas_unseen.jsonl",      test_qas_unseen)
    out["test_qas_seen_dynamic"] = save_jsonl(data_dir / "test_qas_seen_dynamic.jsonl", test_qas_seen_dynamic)
    return out


# ============================================================
# FAISS indexing + retrieval
# ============================================================

def build_faiss_index_from_jsonl(
    index_dir: Union[str, Path],
    jsonl_files: List[Union[str, Path]],
    encoder_model: str = "sentence-transformers/all-mpnet-base-v2",
) -> Path:
    import faiss
    from sentence_transformers import SentenceTransformer

    docs: List[dict] = []
    for p in jsonl_files:
        docs.extend(read_jsonl(p))

    texts = [d["text"] for d in docs]

    enc = SentenceTransformer(encoder_model)
    emb = enc.encode(texts, show_progress_bar=True, convert_to_numpy=True).astype("float32")

    idx = faiss.IndexFlatIP(emb.shape[1])
    idx.add(emb)

    index_dir = Path(index_dir)
    index_dir.mkdir(parents=True, exist_ok=True)

    faiss.write_index(idx, str(index_dir / "index.faiss"))
    save_jsonl(index_dir / "meta.json", docs)

    return index_dir / "index.faiss"


def faiss_retrieve(
    query: str,
    index_dir: Union[str, Path],
    top_k: int = 5,
    encoder_model: str = "sentence-transformers/all-mpnet-base-v2",
) -> List[dict]:
    import faiss
    from sentence_transformers import SentenceTransformer

    enc = SentenceTransformer(encoder_model)
    q_emb = enc.encode([query], convert_to_numpy=True).astype("float32")

    idx = faiss.read_index(str(Path(index_dir) / "index.faiss"))
    metas = read_jsonl(Path(index_dir) / "meta.json")

    D, I = idx.search(q_emb, top_k)
    return [metas[i] for i in I[0]]


# ============================================================
# DATASETS for Axolotl (pretraining + fine-tuning)
# ============================================================

def create_pretraining_dataset(
    input_filename: str = "stable_docs.jsonl",
    output_filename: str = "docs_for_pretrain.jsonl",
    data_dir: Union[str, Path] = DATA_DIR
) -> Path:
    docs = read_jsonl(Path(data_dir) / input_filename)
    return save_jsonl(Path(data_dir) / output_filename, [{"text": d["text"]} for d in docs])


def create_qa_finetune_dataset(
    data_dir: Union[str, Path] = DATA_DIR,
    docs_filename: str = "stable_docs.jsonl",
    qas_filename: str = "train_ft_qas.jsonl",
    output_filename: str = "docs_qa_with_context.jsonl",
    encoder_model: str = "sentence-transformers/all-mpnet-base-v2",
    top_k: int = 3,
) -> Path:
    from sentence_transformers import SentenceTransformer, util

    docs = read_jsonl(Path(data_dir) / docs_filename)
    qas = read_jsonl(Path(data_dir) / qas_filename)

    doc_texts = [d["text"] for d in docs]
    enc = SentenceTransformer(encoder_model)
    doc_embs = enc.encode(doc_texts, convert_to_tensor=True)

    out_rows: List[dict] = []
    for qa in qas:
        q_emb = enc.encode(qa["question"], convert_to_tensor=True)
        sims = util.pytorch_cos_sim(q_emb, doc_embs)[0]
        top_idx = sims.topk(k=min(top_k, len(doc_texts))).indices.tolist()
        context = "\n\n".join(doc_texts[i] for i in top_idx)
        out_rows.append({"id": qa.get("id"), "question": qa["question"], "answer": qa["answer"], "context": context})

    return save_jsonl(Path(data_dir) / output_filename, out_rows)


# ============================================================
# AXOLOTL config generation
# ============================================================

def write_axolotl_configs(
    base_model: str = "mistralai/Mistral-7B-Instruct-v0.2",
    pretrain_jsonl: Union[str, Path] = DATA_DIR / "docs_for_pretrain.jsonl",
    qa_jsonl: Union[str, Path] = DATA_DIR / "docs_qa_with_context.jsonl",
    out_dir: Union[str, Path] = AXOLOTL_CONFIG_DIR,
) -> Dict[str, Path]:

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    pretrain_yaml = f'''
base_model: {base_model}
model_type: mistral
tokenizer_type: mistral
load_in_4bit: true
adapter: qlora
trust_remote_code: true

datasets:
  - path: {Path(pretrain_jsonl).parent.as_posix()}
    type: json
    train_on_split: {Path(pretrain_jsonl).name}
    field_map:
      prompt: text
      response: ""
    prompt_template: chatml

dataset_prepared_path: last_run_pretrain

output_dir: axolotl_outputs/mistral_sft1
sequence_len: 2048
sample_packing: true
pad_to_sequence_len: true

lora_r: 16
lora_alpha: 32
lora_dropout: 0.05
gradient_accumulation_steps: 2
micro_batch_size: 1
num_epochs: 2
optimizer: adamw_bnb_8bit
lr_scheduler: cosine
learning_rate: 2e-4

train_on_inputs: true
group_by_length: false
warmup_steps: 10
save_every: 0
evals_per_epoch: 0
logging_steps: 10
'''
    (out_dir / "config_pretrain.yaml").write_text(pretrain_yaml, encoding="utf-8")

    sft_yaml = f'''
base_model: {base_model}
model_type: mistral
tokenizer_type: mistral
load_in_4bit: true
adapter: qlora
trust_remote_code: true

datasets:
  - path: {Path(qa_jsonl).parent.as_posix()}
    type: json
    train_on_split: {Path(qa_jsonl).name}
    field_map:
      prompt: question
      response: answer
    prompt_template: chatml

dataset_prepared_path: last_run_qa

output_dir: axolotl_outputs/mistral_sft2
sequence_len: 2048
sample_packing: true
pad_to_sequence_len: true

lora_r: 16
lora_alpha: 32
lora_dropout: 0.05
gradient_accumulation_steps: 2
micro_batch_size: 1
num_epochs: 2
optimizer: adamw_bnb_8bit
lr_scheduler: cosine
learning_rate: 1e-4

train_on_inputs: true
group_by_length: false
warmup_steps: 10
save_every: 0
evals_per_epoch: 0
logging_steps: 10
'''
    (out_dir / "config_sft_qa.yaml").write_text(sft_yaml, encoding="utf-8")

    return {
        "pretrain": out_dir / "config_pretrain.yaml",
        "sft": out_dir / "config_sft_qa.yaml",
    }
"""

Path("/content/ft_vs_rag_improved_pipeline.py").write_text(clean_code, encoding="utf-8")
print("✅ English-only module written to /content/ft_vs_rag_improved_pipeline.py")


✅ English-only module written to /content/ft_vs_rag_improved_pipeline.py


In [ ]:
import sys, importlib
sys.path.append("/content")
import ft_vs_rag_improved_pipeline as mod
importlib.reload(mod)

from ft_vs_rag_improved_pipeline import (
    DATA_DIR, INDEX_DIR_FULL, INDEX_DIR_DYNAMIC,
    hotpot_extract, save_jsonl, build_faiss_index_from_jsonl
)

print("OK ✅", DATA_DIR, INDEX_DIR_FULL, INDEX_DIR_DYNAMIC)


OK ✅ /content/ft_vs_rag_multidata/data_hotpot /content/ft_vs_rag_multidata/faiss_index_full /content/ft_vs_rag_multidata/faiss_index_dynamic


In [ ]:
#save common functions in shared drive folder
!mkdir -p "/content/drive/MyDrive/ft_vs_rag_project"
!cp "/content/ft_vs_rag_improved_pipeline.py" "/content/drive/MyDrive/ft_vs_rag_project/"
print("✅ קובץ נשמר בדרייב!")

✅ קובץ נשמר בדרייב!


# FT vs RAG — CPU Notebook

This notebook prepares the HotpotQA data, partitions it into stable and dynamic sets, and builds FAISS indices. Run it on a machine without GPU/TPU to save costs. Ensure that the `datasets`, `transformers`, and `sentence_transformers` packages are installed.

In [ ]:
import sys, importlib
sys.path.append("/content/drive/MyDrive/ft_vs_rag_project")

from ft_vs_rag_improved_pipeline import (
    DATA_DIR, INDEX_DIR_FULL, INDEX_DIR_DYNAMIC,
    hotpot_extract, save_jsonl, make_dynamic_split,
    build_faiss_index_from_jsonl
)

print("OK ✅", DATA_DIR)



OK ✅ /content/ft_vs_rag_multidata/data_hotpot


In [ ]:
# Extract training and validation splits
docs_train, qas_train = hotpot_extract('train', 'distractor')
docs_dev, qas_dev = hotpot_extract('validation', 'distractor')

# Partition documents into stable (train split) and dynamic (validation split not in train)
train_titles = {d['title'] for d in docs_train}
stable_docs = list(docs_train)
dynamic_docs = [d for d in docs_dev if d['title'] not in train_titles]

# Partition QA pairs
train_ft_qas = [q for q in qas_train if all(t in train_titles for t in q['supporting_docs'])]
test_dynamic_qas = [q for q in qas_dev if all(t not in train_titles for t in q['supporting_docs'])]
test_stable_qas = [q for q in qas_dev if all(t in train_titles for t in q['supporting_docs'])]

# Save splits to JSONL files
save_jsonl(DATA_DIR / 'stable_docs.jsonl', stable_docs)
save_jsonl(DATA_DIR / 'dynamic_docs.jsonl', dynamic_docs)
save_jsonl(DATA_DIR / 'train_ft_qas.jsonl', train_ft_qas)
save_jsonl(DATA_DIR / 'test_dynamic_qas.jsonl', test_dynamic_qas)
save_jsonl(DATA_DIR / 'test_stable_qas.jsonl', test_stable_qas)

print(f'Saved {len(stable_docs)} stable docs and {len(dynamic_docs)} dynamic docs.')
print(f'Train QA examples: {len(train_ft_qas)} | Test dynamic QA: {len(test_dynamic_qas)} | Test stable QA: {len(test_stable_qas)}')

In [ ]:
# Build FAISS indices (full and dynamic-only). These calls may take a few minutes.
build_faiss_index_from_jsonl(INDEX_DIR_FULL, [DATA_DIR / 'stable_docs.jsonl', DATA_DIR / 'dynamic_docs.jsonl'])
build_faiss_index_from_jsonl(INDEX_DIR_DYNAMIC, [DATA_DIR / 'dynamic_docs.jsonl'])

In [ ]:
# ================================================================
# Imports
# ================================================================
from ft_vs_rag_improved_pipeline import (
    DATA_DIR, INDEX_DIR_FULL, INDEX_DIR_DYNAMIC,
    hotpot_extract, save_jsonl, make_dynamic_split,
    build_faiss_index_from_jsonl
)

# ================================================================
# 1. Extract HotpotQA training and validation sets
# ================================================================
docs_train, qas_train = hotpot_extract("train", "distractor")
docs_dev,   qas_dev   = hotpot_extract("validation", "distractor")

# ================================================================
# 2. Define stable (train) and dynamic (validation not in train)
# ================================================================
train_titles = {d["title"] for d in docs_train}

stable_docs  = list(docs_train)
dynamic_docs = [d for d in docs_dev if d["title"] not in train_titles]

# ================================================================
# 3. Partition QA pairs according to doc visibility
#    (these are used mostly for reporting/comparison)
# ================================================================
train_ft_qas     = [q for q in qas_train if all(t in train_titles for t in q["supporting_docs"])]
test_dynamic_qas = [q for q in qas_dev   if all(t not in train_titles for t in q["supporting_docs"])]
test_stable_qas  = [q for q in qas_dev   if all(t in train_titles for t in q["supporting_docs"])]

# ================================================================
# 4. Save initial splits
# ================================================================
save_jsonl(DATA_DIR / "stable_docs.jsonl", stable_docs)
save_jsonl(DATA_DIR / "dynamic_docs.jsonl", dynamic_docs)
save_jsonl(DATA_DIR / "train_ft_qas.jsonl", train_ft_qas)
save_jsonl(DATA_DIR / "test_dynamic_qas.jsonl", test_dynamic_qas)  # optional for reporting
save_jsonl(DATA_DIR / "test_stable_qas.jsonl",  test_stable_qas)

print(f"Saved {len(stable_docs)} stable docs and {len(dynamic_docs)} dynamic docs.")
print(f"Training QA: {len(train_ft_qas)} | Dev dynamic QA: {len(test_dynamic_qas)} | Dev stable QA: {len(test_stable_qas)}")


# ================================================================
# 5. NEW: Create dynamic split: indexed vs. held-out (unseen)
#    Output files:
#      - dynamic_indexed_docs.jsonl  -> indexed by RAG
#      - dynamic_test_docs.jsonl     -> NOT indexed and NOT used in FT
#      - test_qas_unseen.jsonl       -> evaluation set not seen by RAG nor FT
#      - test_qas_seen_dynamic.jsonl -> optional, QAs supported only by indexed docs
# ================================================================
split_paths = make_dynamic_split(
    data_dir=DATA_DIR,
    dynamic_docs_name="dynamic_docs.jsonl",
    dev_qas_name="test_dynamic_qas.jsonl",
    ratio_indexed=0.8,
    seed=42
)

print("Dynamic split completed:")
for k, v in split_paths.items():
    print("   ", k, "->", v)


# ================================================================
# 6. Build FAISS indexes
# ================================================================
# FULL index (stable + dynamic-indexed only)
build_faiss_index_from_jsonl(
    INDEX_DIR_FULL,
    [
        DATA_DIR / "stable_docs.jsonl",
        DATA_DIR / "dynamic_indexed_docs.jsonl"
    ]
)

# DYNAMIC index (only dynamic-indexed docs)
build_faiss_index_from_jsonl(
    INDEX_DIR_DYNAMIC,
    [
        DATA_DIR / "dynamic_indexed_docs.jsonl"
    ]
)

print("FAISS index build complete.")
print(" FULL index = stable + dynamic_indexed")
print(" DYNAMIC index = dynamic_indexed only")

# ================================================================
# The evaluation should use:
#   - test_qas_unseen.jsonl (strict evaluation: docs unseen by FT and RAG)
# ================================================================


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

distractor/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/validation-00000-of-00001.par(…):   0%|          | 0.00/27.5M [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/19138 [00:00<?, ?it/s]

In [ ]:
import sys, importlib
sys.path.append("/content/drive/MyDrive/ft_vs_rag_project")

In [ ]:
# ================================================================
# CPU-friendly setup (optional but recommended on Colab CPU)
# ================================================================
import os, torch
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "2"        # tune to your VM cores (2–4)
torch.set_num_threads(2)

# ================================================================
# Imports
# ================================================================
from ft_vs_rag_improved_pipeline import (
    DATA_DIR, INDEX_DIR_FULL, INDEX_DIR_DYNAMIC,
    hotpot_extract, save_jsonl, make_dynamic_split,
    build_faiss_index_from_jsonl
)

# Choose a lighter embedding model for CPU:
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Chunking params to reduce number of chunks:
CHUNK_SIZE = 768
OVERLAP    = 16

# ================================================================
# 1. Extract HotpotQA training and validation sets (with CPU-friendly chunking)
#    Note: increase sample_size for full runs; keep None for entire split.
# ================================================================
docs_train, qas_train = hotpot_extract(
    split="train",
    variant="distractor",
    sample_size=None,              # for quick debugging use a number (e.g., 2000)
    chunk_size=CHUNK_SIZE,
    overlap=OVERLAP
)
docs_dev, qas_dev = hotpot_extract(
    split="validation",
    variant="distractor",
    sample_size=None,
    chunk_size=CHUNK_SIZE,
    overlap=OVERLAP
)

# ================================================================
# 2. Define stable (train) and dynamic (validation not in train)
# ================================================================
train_titles = {d["title"] for d in docs_train}

stable_docs  = list(docs_train)
dynamic_docs = [d for d in docs_dev if d["title"] not in train_titles]

# ================================================================
# 3. Partition QA pairs according to doc visibility (for reporting)
# ================================================================
train_ft_qas     = [q for q in qas_train if all(t in  train_titles for t in q["supporting_docs"])]
test_dynamic_qas = [q for q in qas_dev   if all(t not in train_titles for t in q["supporting_docs"])]
test_stable_qas  = [q for q in qas_dev   if all(t in  train_titles for t in q["supporting_docs"])]

# ================================================================
# 4. Save initial splits
# ================================================================
save_jsonl(DATA_DIR / "stable_docs.jsonl", stable_docs)
save_jsonl(DATA_DIR / "dynamic_docs.jsonl", dynamic_docs)
save_jsonl(DATA_DIR / "train_ft_qas.jsonl", train_ft_qas)
save_jsonl(DATA_DIR / "test_dynamic_qas.jsonl", test_dynamic_qas)  # optional for reporting
save_jsonl(DATA_DIR / "test_stable_qas.jsonl",  test_stable_qas)

print(f"Saved {len(stable_docs)} stable docs and {len(dynamic_docs)} dynamic docs.")
print(f"Training QA: {len(train_ft_qas)} | Dev dynamic QA: {len(test_dynamic_qas)} | Dev stable QA: {len(test_stable_qas)}")

# ================================================================
# 5. Dynamic split: indexed vs. held-out (unseen)
# ================================================================
split_paths = make_dynamic_split(
    data_dir=DATA_DIR,
    dynamic_docs_name="dynamic_docs.jsonl",
    dev_qas_name="test_dynamic_qas.jsonl",
    ratio_indexed=0.8,
    seed=42
)
print("Dynamic split completed:")
for k, v in split_paths.items():
    print("   ", k, "->", v)

# ================================================================
# 6. Build FAISS indexes (CPU-friendly: use MiniLM model)
#    FULL index = stable + dynamic_indexed
#    DYNAMIC index = dynamic_indexed only
# ================================================================
build_faiss_index_from_jsonl(
    INDEX_DIR_FULL,
    [
        DATA_DIR / "stable_docs.jsonl",
        DATA_DIR / "dynamic_indexed_docs.jsonl"
    ],
    encoder_model=EMBED_MODEL
)

build_faiss_index_from_jsonl(
    INDEX_DIR_DYNAMIC,
    [
        DATA_DIR / "dynamic_indexed_docs.jsonl"
    ],
    encoder_model=EMBED_MODEL
)

print("FAISS index build complete (CPU).")
print(" FULL index = stable + dynamic_indexed")
print(" DYNAMIC index = dynamic_indexed only")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

distractor/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/validation-00000-of-00001.par(…):   0%|          | 0.00/27.5M [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/17305 [00:00<?, ?it/s]

In [ ]:
# ================================================================
# RAG sanity checks (manual + light automated)
# ================================================================
from pathlib import Path
import os
from collections import Counter

from ft_vs_rag_improved_pipeline import (
    DATA_DIR, INDEX_DIR_FULL, INDEX_DIR_DYNAMIC,
    read_jsonl, faiss_retrieve
)

# Use the SAME embedding model as used to build your index
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# ------------------------------------------------
# Helpers
# ------------------------------------------------
def index_files_exist(index_dir: Path) -> bool:
    return (index_dir / "index.faiss").exists() and (index_dir / "meta.json").exists()

def load_index_meta(index_dir: Path):
    meta = read_jsonl(index_dir / "meta.json")
    return meta

def titles_from_docs(docs):
    return { (d.get("title") or "").strip() for d in docs if d.get("title") }

def check_file_exists(p: Path):
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {p}")

def print_header(txt: str):
    print("\n" + "=" * 70)
    print(txt)
    print("=" * 70)

# ------------------------------------------------
# 0) Files presence & basic shapes
# ------------------------------------------------
def check_files_presence():
    print_header("CHECK 0: files presence")
    for name in [
        "stable_docs.jsonl",
        "dynamic_docs.jsonl",
        "dynamic_indexed_docs.jsonl",
        "dynamic_test_docs.jsonl",
        "train_ft_qas.jsonl",
        "test_dynamic_qas.jsonl",
        "test_stable_qas.jsonl",
        "test_qas_unseen.jsonl"
    ]:
        p = DATA_DIR / name
        check_file_exists(p)
        print(f"OK: {name} ({len(read_jsonl(p))} rows)")

# ------------------------------------------------
# 1) Index integrity
# ------------------------------------------------
def check_index_integrity():
    print_header("CHECK 1: index integrity")

    for label, idx_dir in [("HYBRID (dynamic)", INDEX_DIR_DYNAMIC), ("FULL", INDEX_DIR_FULL)]:
        print(f"\n{label} index @ {idx_dir}")
        if not index_files_exist(idx_dir):
            raise FileNotFoundError(f"Index files missing in {idx_dir}")
        meta = load_index_meta(idx_dir)
        print(f"OK: meta.json size = {len(meta)}")
        # Basic non-empty text check
        empty = sum(1 for r in meta if not r.get("text"))
        print(f"Non-empty text rows = {len(meta) - empty} / {len(meta)}")

# ------------------------------------------------
# 2) Leakage checks
#    - Ensure held-out dynamic titles are NOT in any index meta
#    - Ensure test_qas_unseen truly rely on held-out titles only
# ------------------------------------------------
def check_leakage():
    print_header("CHECK 2: leakage")

    dyn_indexed = read_jsonl(DATA_DIR / "dynamic_indexed_docs.jsonl")
    dyn_test    = read_jsonl(DATA_DIR / "dynamic_test_docs.jsonl")
    test_unseen = read_jsonl(DATA_DIR / "test_qas_unseen.jsonl")

    titles_indexed = titles_from_docs(dyn_indexed)
    titles_heldout = titles_from_docs(dyn_test)

    # a) held-out titles must not appear in any index meta
    for label, idx_dir in [("HYBRID", INDEX_DIR_DYNAMIC), ("FULL", INDEX_DIR_FULL)]:
        meta = load_index_meta(idx_dir)
        meta_titles = titles_from_docs(meta)
        overlap = titles_heldout & meta_titles
        if overlap:
            raise AssertionError(f"[{label}] Leakage: held-out titles present in index: {list(sorted(overlap))[:5]} ...")
        print(f"OK: [{label}] No held-out titles in index")

    # b) test_qas_unseen must be fully supported by held-out titles only
    bad = 0
    for q in test_unseen:
        supp = {(t or "").strip() for t in q.get("supporting_docs", []) if t}
        if not supp or not supp.issubset(titles_heldout):
            bad += 1
    if bad:
        raise AssertionError(f"Unseen QA set not clean: {bad} items not fully held-out")
    print(f"OK: test_qas_unseen clean ({len(test_unseen)} items)")

# ------------------------------------------------
# 3) Retrieval sanity (determinism, shape, non-empty)
# ------------------------------------------------
def check_retrieval_sanity(mode: str = "hybrid", k: int = 5, n: int = 50):
    """
    mode: 'hybrid' (dynamic-only index) | 'full' (stable + dynamic-indexed)
    """
    print_header(f"CHECK 3: retrieval sanity [{mode.upper()}]")

    index_dir = INDEX_DIR_DYNAMIC if mode == "hybrid" else INDEX_DIR_FULL

    # Pick a small question set:
    # - For 'hybrid' sanity, dynamic-seen set is a reasonable smoke test if you saved it.
    #   If not, use test_dynamic_qas.jsonl (less strict).
    # - For 'full' sanity, test_stable_qas.jsonl is fine as a smoke test.
    if mode == "hybrid" and (DATA_DIR / "test_qas_seen_dynamic.jsonl").exists():
        qas = read_jsonl(DATA_DIR / "test_qas_seen_dynamic.jsonl")
        source_name = "test_qas_seen_dynamic.jsonl"
    elif mode == "hybrid":
        qas = read_jsonl(DATA_DIR / "test_dynamic_qas.jsonl")  # fallback
        source_name = "test_dynamic_qas.jsonl"
    else:
        qas = read_jsonl(DATA_DIR / "test_stable_qas.jsonl")
        source_name = "test_stable_qas.jsonl"

    qas = qas[:n]
    print(f"Using {source_name} ({len(qas)} items)")

    # Determinism: run retrieval twice for first few queries and compare doc_ids
    mismatches = 0
    for q in qas[:min(10, len(qas))]:
        qtext = q["question"]
        hits1 = faiss_retrieve(qtext, index_dir, top_k=k, encoder_model=EMBED_MODEL)
        hits2 = faiss_retrieve(qtext, index_dir, top_k=k, encoder_model=EMBED_MODEL)
        ids1 = [h.get("doc_id") for h in hits1]
        ids2 = [h.get("doc_id") for h in hits2]
        if ids1 != ids2:
            mismatches += 1

        # Non-empty checks
        assert all(h.get("text") for h in hits1), "Empty text in retrieved hits"
        assert len(hits1) == k or len(hits1) == len(load_index_meta(index_dir)), "Top-k size mismatch"

    if mismatches:
        print(f"Warning: {mismatches} / {min(10, len(qas))} queries had non-deterministic top-k order (may be OK).")
    else:
        print("OK: deterministic top-k for sampled queries")

# ------------------------------------------------
# 4) Title-level Hit@K (quick proxy for retrieval quality)
#    Measures fraction of QAs whose supporting titles appear in top-k.
# ------------------------------------------------
def title_hit_at_k(mode: str = "hybrid", k: int = 5, n: int = 200):
    print_header(f"CHECK 4: title Hit@{k} [{mode.upper()}]")

    index_dir = INDEX_DIR_DYNAMIC if mode == "hybrid" else INDEX_DIR_FULL

    # Choose evaluation set:
    # - For strict evaluation: test_qas_unseen.jsonl (expect lower hit rate in hybrid).
    # - For hybrid smoke: test_qas_seen_dynamic.jsonl (if available).
    # - For full smoke: test_stable_qas.jsonl.
    if (DATA_DIR / "test_qas_unseen.jsonl").exists():
        qas = read_jsonl(DATA_DIR / "test_qas_unseen.jsonl")
        source = "test_qas_unseen.jsonl"
    else:
        # Fallback if unseen set not present
        source = "test_dynamic_qas.jsonl" if mode == "hybrid" else "test_stable_qas.jsonl"
        qas = read_jsonl(DATA_DIR / source)

    qas = qas[:n]
    print(f"Using {source} ({len(qas)} items)")

    hits = 0
    for q in qas:
        supp_titles = {(t or "").strip() for t in q.get("supporting_docs", []) if t}
        if not supp_titles:
            # if supporting_docs not available in your saved QA, skip metric
            continue
        retrieved = faiss_retrieve(q["question"], index_dir, top_k=k, encoder_model=EMBED_MODEL)
        ret_titles = {(r.get("title") or "").strip() for r in retrieved if r.get("title")}
        if supp_titles & ret_titles:
            hits += 1
    denom = len(qas) if len(qas) else 1
    print(f"Title Hit@{k}: {hits}/{denom} = {hits/denom:.3f}")

# ------------------------------------------------
# Run all checks
# ------------------------------------------------
def run_all_rag_checks():
    check_files_presence()
    check_index_integrity()
    check_leakage()
    check_retrieval_sanity(mode="hybrid", k=5, n=50)
    check_retrieval_sanity(mode="full",   k=5, n=50)
    title_hit_at_k(mode="hybrid", k=5, n=200)
    title_hit_at_k(mode="full",   k=5, n=200)

# Execute:
run_all_rag_checks()
